# Baseline Comparison — DCGAN vs cWGAN-GP vs StyleGAN2-ADA

Reads the **saved artifacts** (`history.csv`, `final_metrics.json/csv`, `config.json`)
of the three baselines and builds a comparison table + publication plots
(font 20, dpi 300). **No models are retrained here.**

**Fairness note:** the *held-out* FID/KID (recomputed against the same test set
with the same `batik_gan.metrics` evaluator) are directly comparable across all
three models. StyleGAN2-ADA's native `fid50k_full` (training reals) is shown
separately and must NOT be compared directly with the held-out numbers.


## 1. Setup

In [ ]:

import os, sys, json, subprocess
REPO_URL = "https://github.com/sid-2k6/Textile_Pattern_GAN.git"
REPO_DIR = "/content/Textile_Pattern_GAN"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=False)

import numpy as np, pandas as pd
from batik_gan import viz
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception:
    pass

# EDIT to match where the baselines wrote their outputs:
OUTPUT_ROOT = "/content/drive/MyDrive/Textile_Pattern_GAN/outputs"
MODELS = ["DCGAN", "cWGAN_GP", "StyleGAN2_ADA"]
print("Reading outputs from", OUTPUT_ROOT)


## 2. Comparison table

In [ ]:

# =====================================================================
# BUILD COMPARISON TABLE from final_metrics.json / history.csv / config.json
# =====================================================================
def load_json(p):
    return json.load(open(p)) if os.path.isfile(p) else {}

def num(x):
    try:
        return float(x)
    except Exception:
        return float("nan")

rows = []
histories = {}
for m in MODELS:
    base = os.path.join(OUTPUT_ROOT, m)
    fm = load_json(os.path.join(base, "final_metrics.json"))
    cfg = load_json(os.path.join(base, "config.json"))
    hp = os.path.join(base, "history.csv")
    hist = pd.read_csv(hp) if os.path.isfile(hp) else pd.DataFrame()
    histories[m] = hist
    params = fm.get("generator_parameters", cfg.get("generator_parameters"))
    rows.append({
        "Model": m,
        "Best FID (held-out)": num(fm.get("best_fid")),
        "Final FID (held-out)": num(fm.get("final_fid")),
        "Best KID": num(fm.get("best_kid")),
        "Final KID": num(fm.get("final_kid")),
        "Best Epoch/Tick": fm.get("best_epoch", fm.get("best_tick")),
        "Final Gen Loss": num(fm.get("final_generator_loss")),
        "Final Disc/Critic Loss": num(fm.get("final_discriminator_loss", fm.get("final_critic_loss"))),
        "Training Time (s)": num(fm.get("training_time_sec")),
        "Generator Params": params,
        "Peak GPU Mem (MB)": num(fm.get("peak_gpu_memory_mb")),
        "Native FID50k (SG2 only)": num(fm.get("best_native_fid50k_full")),
    })
table = pd.DataFrame(rows)
pd.set_option("display.max_columns", None, "display.width", 200)
print(table.to_string(index=False))
os.makedirs(os.path.join(OUTPUT_ROOT, "comparison"), exist_ok=True)
table.to_csv(os.path.join(OUTPUT_ROOT, "comparison", "baseline_comparison.csv"), index=False)


## 3. Comparison plots

In [ ]:

# =====================================================================
# PUBLICATION COMPARISON PLOTS (font 20, dpi 300)
# =====================================================================
plt = viz.set_pub_style()
CMP_DIR = os.path.join(OUTPUT_ROOT, "comparison"); os.makedirs(CMP_DIR, exist_ok=True)

# 1) FID vs epoch overlay for the per-epoch-eval models (DCGAN, cWGAN-GP)
fig, ax = plt.subplots(figsize=(10, 7))
plotted = False
for m in ["DCGAN", "cWGAN_GP"]:
    h = histories.get(m, pd.DataFrame())
    if len(h) and "fid" in h and h["fid"].notna().any():
        ax.plot(h["epoch"], h["fid"], marker="o", markersize=3, linewidth=2, label=m)
        plotted = True
ax.set_title("Held-out FID vs Epoch"); ax.set_xlabel("epoch"); ax.set_ylabel("FID (lower better)")
if plotted: ax.legend()
ax.grid(True, alpha=0.3); fig.tight_layout()
fig.savefig(os.path.join(CMP_DIR, "fid_vs_epoch.png"), dpi=300, bbox_inches="tight"); plt.close(fig)

# 2) bar charts: final held-out FID / KID / params / training time
def bar(metric_col, title, ylabel, fname, logy=False):
    sub = table.dropna(subset=[metric_col]) if metric_col in table else pd.DataFrame()
    if not len(sub):
        print("skip", title); return
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.bar(sub["Model"], sub[metric_col])
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel("Model")
    if logy: ax.set_yscale("log")
    for i, v in enumerate(sub[metric_col]):
        ax.text(i, v, f"{v:.3g}", ha="center", va="bottom")
    ax.grid(True, axis="y", alpha=0.3); fig.tight_layout()
    fig.savefig(os.path.join(CMP_DIR, fname), dpi=300, bbox_inches="tight"); plt.close(fig)

bar("Final FID (held-out)", "Final held-out FID by model", "FID", "final_fid_bar.png")
bar("Final KID", "Final held-out KID by model", "KID", "final_kid_bar.png")
bar("Generator Params", "Generator parameters by model", "params", "params_bar.png", logy=True)
bar("Training Time (s)", "Training time by model", "seconds", "training_time_bar.png")
print("Saved comparison plots to", CMP_DIR)


## 4. Notes

In [ ]:

# =====================================================================
# INTERPRETATION NOTES (printed; no fabricated values)
# =====================================================================
print("Comparison artifacts written to", os.path.join(OUTPUT_ROOT, "comparison"))
print("- Held-out FID/KID use the SAME evaluator + test set for all 3 models -> comparable.")
print("- StyleGAN2-ADA native fid50k_full uses TRAINING reals -> shown separately, not comparable.")
print("- Any NaN cells mean that model has not been trained/evaluated yet (not zero).")
print("- Generator 'accuracy/precision/recall/F1' are intentionally absent (undefined for GANs).")
